In [40]:
from app_recouple import *
import pandas as pd

In [41]:
db = pd.read_csv('predictions_alexnet.txt')
ars = pd.read_csv('ars.csv')
flare_actual = pd.read_csv('Flares.csv')

In [42]:
def get_flare(date_param, df):
    """
    Checks if there are any records in the dataframe where isFlare is True
    and largestEventDate is within the date_param and 24 hours after.

    Parameters:
    date_param (datetime): The date to check against.
    df (DataFrame): The dataframe containing the records.

    Returns:
    DataFrame: A dataframe containing the matching records.
    """
    # Convert the date_param to datetime if it is not already
    if not isinstance(date_param, pd.Timestamp):
        date_param = pd.to_datetime(date_param)

    # Define the end date (24 hours after date_param)
    end_date = date_param + timedelta(hours=24)
    end_date = pd.to_datetime(end_date).tz_localize(None)
    date_param = pd.to_datetime(date_param).tz_localize(None)

    # print(date_param.values[0])
    # print(end_date.values[0])
    # print(type(end_date.values[0]))

    # Filter the dataframe
    result = df[(df['Is_Flare'] == True) & 
                (pd.to_datetime(df['Largest Event Date']) >= pd.to_datetime(date_param)) & 
                (pd.to_datetime(df['Largest Event Date']) < pd.to_datetime(end_date))]


    # result = df[(df['Is_Flare'] == True) & 
    #             (df['Largest Event Date'] >= date_param) & 
    #             (df['Largest Event Date'] < end_date)]

    return result

In [43]:
# Find the row with the specific timestamp in the DataFrame
def get_metrics(row,plage,events):
    try:
        guidedgradcam = retrieve_npy(row['local_request_date'],media='vgg',root='guidedgradcam')
        original = retrieve_npy(row['local_request_date'],media='vgg',root='original')
        ar_names,ar_lat,ar_lon = retrieve_ar(row['local_request_date'],
        plage=plage ,
        events=events,
        media='vgg'
        )

        actual_pred = get_flare(row['obs_date'],flare_actual)
        # if actual_pred.shape[0] >= 1:
        #     actual_pred = 1
        # else:
        #     actual_pred = 0

        pix_list = [post_hoc_explanation.convert_to_pix(float(x[1]),float(x[0])) for x in zip(ar_lat,ar_lon)]
        flares = [item for item in list(zip(ar_names,pix_list))]

        # Analyze attention map
        bounding_hulls_img, distances_df, score, ratio = post_hoc_explanation.explain(guidedgradcam, flares, -20, 20, 5, 40, 30, 50, 2, 10, numpy=True)
    except Exception as e:
        return None, None, None, actual_pred.shape[0]
    return  distances_df.to_dict(orient='list'), score, ratio,actual_pred.shape[0]

In [44]:
db[['distances_df_all', 'score_all', 'ratio_all','actual']] = db.apply(
    lambda row: pd.Series(get_metrics(row, True, True)), axis=1
    
)

(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)
(512, 512)

In [45]:
# db.to_csv('processed_alexnet.csv',index=False)

In [46]:
# import pandas as pd

# Assuming df is your DataFrame
# Step 1: Convert local_request_date to datetime format
db['local_request_date'] = pd.to_datetime(db['local_request_date'])
# print(db.shape)

# Step 2: Filter out rows where ratio is NaN
df_filtered = db.dropna(subset=['score_all']).reset_index(drop=True)
# print(df_filtered.shape)

# Step 3: Extract year, month, and day
df_filtered['year'] = df_filtered['local_request_date'].dt.year
df_filtered['month'] = df_filtered['local_request_date'].dt.month
df_filtered['day'] = df_filtered['local_request_date'].dt.day

# Step 4: Group by year and month, then count the number of unique days
result = df_filtered.groupby(['year', 'month'])['day'].nunique().reset_index()

# Rename the 'day' column to 'unique_days' for clarity
result = result.rename(columns={'day': 'unique_days'})

# print(result)


In [47]:
df_filtered['actual_pred']=df_filtered['actual'].apply(lambda x: 1 if x >= 1.0 else 0)
df_filtered['actual'].value_counts()
df_filtered['score_all_bound'] = df_filtered['score_all']/400
# df_filtered[(df_filtered.actual_pred != 0) & (df_filtered.actual <1)][['actual_pred','actual']]

In [48]:
# import pandas as pd

# Assuming df is your DataFrame with 'scored_probabilities' and 'actual' fields
# Define the threshold
threshold = 0.5

# Step 1: Get predicted labels based on the threshold
df_filtered['predicted'] = df_filtered['flare_probability'].apply(lambda x: 1 if abs(x) >= threshold else 0)

# Step 2: Classify each record as FP, TP, FN, or TN
def classify(row):
    if row['predicted'] == 1 and row['actual_pred'] == 1:
        return 'TP'
    elif row['predicted'] == 1 and row['actual_pred'] == 0:
        return 'FP'
    elif row['predicted'] == 0 and row['actual_pred'] == 1:
        return 'FN'
    elif row['predicted'] == 0 and row['actual_pred'] == 0:
        return 'TN'

df_filtered['classification'] = df_filtered.apply(classify, axis=1)

# Display the DataFrame with the new column
print(df_filtered['classification'].value_counts())


classification
TN    936
FP    191
TP    128
FN     86
Name: count, dtype: int64


In [49]:
# Group by classification and calculate descriptive statistics for the ratio field
stats = df_filtered.groupby('classification')['ratio_all'].describe()
stats

,count,mean,std,min,25%,50%,75%,max
classification,,,,,,,,
FN,86.0,0.787674,0.243169,0.00,0.67,0.80,1.0,1.0
FP,191.0,0.647173,0.288501,0.14,0.33,0.67,1.0,1.0
TN,936.0,0.795342,0.258482,0.00,0.67,1.00,1.0,1.0
TP,128.0,0.684297,0.278242,0.17,0.40,0.71,1.0,1.0


In [50]:
import numpy as np
from sklearn.metrics import classification_report

# Generate classification report
report = classification_report(df_filtered['actual_pred'], df_filtered['predicted'], target_names=['Non Flare', 'Flare'])

# Print classification report
print("Classification Report:")
print(report)

Classification Report:
              precision    recall  f1-score   support

   Non Flare       0.92      0.83      0.87      1127
       Flare       0.40      0.60      0.48       214

    accuracy                           0.79      1341
   macro avg       0.66      0.71      0.68      1341
weighted avg       0.83      0.79      0.81      1341



In [51]:
df_filtered.columns

Index(['source_date', 'obs_date', 'raw_filename', 'noaa_ar_filename',
       'local_request_date', 'error', 'flare_probability',
       'non_flare_probability', 'explanation', 'distances_df_all', 'score_all',
       'ratio_all', 'actual', 'year', 'month', 'day', 'actual_pred',
       'score_all_bound', 'predicted', 'classification'],
      dtype='object')

In [52]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create a box plot to visualize the ratio field for each classification category
plt.figure(figsize=(10, 6))
sns.boxplot(x='classification', y='score_all', data=df_filtered)
plt.title('Distribution of Ratio Field by Classification Category')
plt.xlabel('Classification')
plt.ylabel('Ratio')
plt.show()


C:\Users\yemi\AppData\Local\Temp\ipykernel_20772\347541184.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [53]:
import matplotlib.pyplot as plt

# Create box plots
plt.figure(figsize=(10, 6))
df_filtered.boxplot(column='ratio_all', by='classification', grid=False)
plt.title('Distribution of Ratio Field by Classification Category')
plt.suptitle('')  # Suppress the default title
plt.xlabel('Classification')
plt.ylabel('Ratio')
plt.show()


C:\Users\yemi\AppData\Local\Temp\ipykernel_20772\3031245626.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [54]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Assuming df is your DataFrame with 'score_all', 'ratio_all', and 'classification' fields
data = df_filtered[['score_all_bound', 'ratio_all']]

# Handle missing values if any
data = data.dropna()

# Standardize the data
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# Using the Elbow Method to determine the optimal number of clusters
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=42)
    kmeans.fit(data_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss)
plt.title('Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()

# Using Silhouette Score to determine the optimal number of clusters
silhouette_scores = []
for i in range(2, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=42)
    kmeans.fit(data_scaled)
    score = silhouette_score(data_scaled, kmeans.labels_)
    silhouette_scores.append(score)

plt.figure(figsize=(10, 6))
plt.plot(range(2, 11), silhouette_scores)
plt.title('Silhouette Score Method')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette Score')
plt.show()

# Performing K-Means clustering with the chosen number of clusters (let's assume 4 for this example)
kmeans = KMeans(n_clusters=2, init='k-means++', max_iter=300, n_init=10, random_state=42)
df_filtered['cluster'] = kmeans.fit_predict(data_scaled)

# Visualizing the clusters
plt.figure(figsize=(12, 8))
sns.scatterplot(x='score_all_bound', y='ratio_all', hue='classification', style='cluster', palette='deep', data=df_filtered)
plt.title('Clustering of Data Points with Color Coding Based on Classification')
plt.xlabel('Score All')
plt.ylabel('Ratio All')
plt.legend(loc='best')
plt.show()


C:\Users\yemi\AppData\Local\Temp\ipykernel_20772\3882412094.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\yemi\AppData\Local\Temp\ipykernel_20772\3882412094.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\yemi\AppData\Local\Temp\ipykernel_20772\3882412094.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [55]:
df_filtered.to_csv('processed_alexnet.csv',index=False)

In [2]:
import pandas as pd


res=pd.read_csv('processed_resnet.csv').drop_duplicates(keep='first')
# vgg=pd.read_csv('processed_vgg.csv').drop_duplicates(keep='first')
alex=pd.read_csv('processed_alexnet.csv').drop_duplicates(keep='first')

In [3]:
merged = pd.merge(res, alex, on=['source_date','obs_date','raw_filename','local_request_date','year','month','day'], how= 'inner', suffixes=('_res', '_alex'))
# merged = pd.merge(merged, alex, on=['source_date','obs_date','raw_filename','local_request_date','year','month','day'], how= 'inner', suffixes=('_other', '_alex'))
# alex[['source_date','obs_date','raw_filename','local_request_date','year','month','day']]


In [4]:
merged

,source_date,obs_date,raw_filename,noaa_ar_filename_res,local_request_date,error_res,flare_probability_res,non_flare_probability_res,explanation_res,distances_res,...,error_alex,flare_probability_alex,non_flare_probability_alex,explanation_alex,distances_alex,score_alex,ratio_alex,actual_alex,score_f_alex,ratio_f_alex
0,2015-01-01 00:18:09,2014-12-31T23:59:39.10Z,2014_12_31__23_59_39_105__SDO_HMI_HMI_magnetog...,2024_7_16_0030_UTC.txt,2015-01-01 00:00:00,NaN,-0.908135,1.480420,NaN,"{'Flare': [12248, 12251, 12252, 12253], 'Close...",...,NaN,0.314645,0.685355,NaN,"{'Flare': [12248, 12251, 12252, 12253], 'Close...",13.33,0.75,0,0.00,1.0
1,2015-01-01 04:21:05,2015-01-01T03:59:39.20Z,2015_01_01__03_59_39_205__SDO_HMI_HMI_magnetog...,2024_7_16_0030_UTC.txt,2015-01-01 04:00:00,NaN,-0.757555,1.326106,NaN,"{'Flare': [12248, 12251, 12252, 12253], 'Close...",...,NaN,0.335541,0.664459,NaN,"{'Flare': [12248, 12251, 12252, 12253], 'Close...",0.64,0.75,0,0.00,1.0
2,2015-01-01 08:20:03,2015-01-01T07:59:39.30Z,2015_01_01__07_59_39_305__SDO_HMI_HMI_magnetog...,2024_7_16_0030_UTC.txt,2015-01-01 08:00:00,NaN,-0.667645,1.124752,NaN,"{'Flare': [12248, 12251, 12252, 12253], 'Close...",...,NaN,0.589686,0.410314,NaN,"{'Flare': [12248, 12251, 12252, 12253], 'Close...",1.14,0.50,0,0.00,1.0
3,2015-01-01 12:18:08,2015-01-01T11:59:39.20Z,2015_01_01__11_59_39_205__SDO_HMI_HMI_magnetog...,2024_7_16_0030_UTC.txt,2015-01-01 12:00:00,NaN,-0.681340,1.266520,NaN,"{'Flare': [12248, 12251, 12252, 12253], 'Close...",...,NaN,0.607903,0.392097,NaN,"{'Flare': [12248, 12251, 12252, 12253], 'Close...",1.12,0.75,0,0.00,1.0
4,2015-01-01 16:20:06,2015-01-01T15:59:39.10Z,2015_01_01__15_59_39_105__SDO_HMI_HMI_magnetog...,2024_7_16_0030_UTC.txt,2015-01-01 16:00:00,NaN,-0.505055,1.030208,NaN,"{'Flare': [12248, 12251, 12252, 12253], 'Close...",...,NaN,0.859357,0.140643,NaN,"{'Flare': [12248, 12251, 12252, 12253], 'Close...",9.18,0.50,0,27.66,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2149,2015-12-26T17:18:17,2015-12-26 15:59:38,2015_12_26__15_59_38_20__SDO_HMI_HMI_magnetogr...,2024_7_16_0030_UTC.txt,2015-12-26 16:00:00,NaN,-0.816328,1.272536,NaN,"{'Flare': [12470, 12472, 12473], 'Closest Hull...",...,NaN,0.150792,0.849208,NaN,"{'Flare': [12470, 12472, 12473], 'Closest Hull...",62.95,0.33,0,0.45,0.5
2150,2015-12-26T21:17:16,2015-12-26 19:59:38,2015_12_26__19_59_38_10__SDO_HMI_HMI_magnetogr...,2024_7_16_0030_UTC.txt,2015-12-26 20:00:00,NaN,-0.775106,1.197879,NaN,"{'Flare': [12470, 12472, 12473], 'Closest Hull...",...,NaN,0.268427,0.731573,NaN,"{'Flare': [12470, 12472, 12473], 'Closest Hull...",56.22,0.33,0,1.12,0.5
2151,2015-12-27T06:34:19,2015-12-26 23:02:38,2015_12_26__23_02_38_20__SDO_HMI_HMI_magnetogr...,2024_7_16_0030_UTC.txt,2015-12-27 00:00:00,NaN,-0.511038,1.035648,NaN,"{'Flare': [12472, 12473], 'Closest Hull': [0, ...",...,NaN,0.940105,0.059895,NaN,"{'Flare': [12472, 12473], 'Closest Hull': [1, ...",28.79,0.50,0,28.79,0.5
2152,2015-12-27T07:29:11,2015-12-27 06:09:23,2015_12_27__06_09_23_30__SDO_HMI_HMI_magnetogr...,2024_7_16_0030_UTC.txt,2015-12-27 04:00:00,NaN,0.191253,0.378800,NaN,"{'Flare': [12472, 12473], 'Closest Hull': [1, ...",...,NaN,0.291853,0.708147,NaN,"{'Flare': [12472, 12473], 'Closest Hull': [3, ...",0.00,1.00,0,0.00,1.0


In [5]:
merged.to_csv('merged_predictions.txt',index=False)

In [65]:
merged.columns

Index(['source_date', 'obs_date', 'raw_filename', 'noaa_ar_filename_res',
       'local_request_date', 'error_res', 'flare_probability_res',
       'non_flare_probability_res', 'explanation_res', 'distances_df_all_res',
       'score_all_res', 'ratio_all_res', 'actual_res', 'year', 'month', 'day',
       'actual_pred_res', 'score_all_bound_res', 'predicted_res',
       'classification_res', 'cluster_res', 'noaa_ar_filename_vgg',
       'error_vgg', 'flare_probability_vgg', 'non_flare_probability_vgg',
       'explanation_vgg', 'distances_df_all_vgg', 'score_all_vgg',
       'ratio_all_vgg', 'actual_vgg', 'actual_pred_vgg', 'score_all_bound_vgg',
       'predicted_vgg', 'classification_vgg', 'cluster_vgg'],
      dtype='object')